In [1]:
import evidently
evidently.__version__

'0.6.6'

In [14]:
import joblib
import pandas as pd
from src.data_preprocessing import Cleaner
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset , DataQualityPreset , TargetDriftPreset 
from evidently import ColumnMapping
import warnings
warnings.filterwarnings("ignore")


In [23]:
model = joblib.load('models/model.pkl')

reference = pd.read_csv("data/train.csv")
current = pd.read_csv("data/test.csv")
production = pd.read_csv("data/production.csv")

In [24]:
production.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 99 non-null     int64  
 1   Gender             92 non-null     object 
 2   Age                93 non-null     float64
 3   HasDrivingLicense  91 non-null     float64
 4   RegionID           90 non-null     float64
 5   Switch             51 non-null     float64
 6   VehicleAge         83 non-null     object 
 7   PastAccident       59 non-null     object 
 8   AnnualPremium      99 non-null     object 
 9   SalesChannelID     99 non-null     int64  
 10  DaysSinceCreated   99 non-null     int64  
 11  Result             99 non-null     int64  
dtypes: float64(4), int64(4), object(4)
memory usage: 9.4+ KB


In [25]:
cleaner = Cleaner()
reference = cleaner.clean_data(reference)
reference['prediction'] = model.predict(reference.iloc[:,:-1])

current = cleaner.clean_data(current)
current['prediction'] = model.predict(current.iloc[:,:-1])



In [26]:
target = "Result"
prediction = 'prediction'
numerical_features = ['Age','AnnualPremium','HasDrivingLicense','RegionID','Switch']
categorical_features = ['Gender','PastAccident']
column_mapping = ColumnMapping()

column_mapping.target = target
column_mapping.prediction = prediction
column_mapping.numerical_features = numerical_features
column_mapping.categorical_features = categorical_features

In [27]:
data_drift_report = Report(metrics = [
    DataDriftPreset(),
    DataQualityPreset(),
    TargetDriftPreset()
])

data_drift_report.run(reference_data = reference, current_data = current , column_mapping = column_mapping)
data_drift_report
data_drift_report.save_html("test_drift.html")
